# 04. FAISSインデックス構築

staging の統合データから FAISS インデックスを構築して `data/staging/faiss/` に保存する。
**本番インデックスへの書き込みは行わない。** 本番反映は `05_export_to_fastapi.ipynb` で実施。

| インデックス | ソース | 出力 |
|-----------|--------|------|
| entities | staging/sanctions/sanctions_merged.json | staging/faiss/entities.index |
| matrix_rules | ai_validation DB (現行) | staging/faiss/matrix_rules.index |

In [ ]:
DRY_RUN = False

import sys, json, logging
from pathlib import Path

try:
    BASE
except NameError:
    BASE        = Path("/Users/takehirosato/Desktop/AI_TradeManagement")
    STAGING_DIR = BASE / "data" / "staging"
    sys.path.insert(0, str(BASE / "scripts"))

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
FAISS_OUT = STAGING_DIR / "faiss"
FAISS_OUT.mkdir(parents=True, exist_ok=True)
print(f"DRY_RUN={DRY_RUN}, FAISS_OUT={FAISS_OUT}")

## 1. 制裁エンティティ FAISS 構築

In [ ]:
from pipeline.index.faiss_builder import build_sanctions_index

merged_path = STAGING_DIR / "sanctions" / "sanctions_merged.json"
if not merged_path.exists():
    print("⚠️  sanctions_merged.json が見つかりません。02 を先に実行してください。")
else:
    entities = json.loads(merged_path.read_text())
    print(f"  入力: {len(entities):,} エンティティ")
    result = build_sanctions_index(
        entities=entities,
        output_dir=FAISS_OUT,
        dry_run=DRY_RUN,
    )
    if result:
        ntotal, idx_path, meta_path = result
        print(f"✅ entities.index 構築完了: ntotal={ntotal:,}")
        print(f"   {idx_path}  ({idx_path.stat().st_size:,} bytes)")
    else:
        print("[DRY_RUN] インデックス構築をシミュレーション")

## 2. 規制マトリクス FAISS 構築

In [ ]:
# ai_validation DB から現行の matrix_rules を読み込んでインデックスを再構築
import sqlite3

DB_PATH = BASE / "modules" / "ai_validation" / "app.db"
if not DB_PATH.exists():
    print(f"⚠️  DB not found: {DB_PATH}")
else:
    conn = sqlite3.connect(str(DB_PATH))
    rows = conn.execute(
        "SELECT id, title, requirement_text, usage_criteria_text, "
        "tech_criteria_text, notes, item_no, list_name FROM matrix_rules"
    ).fetchall()
    conn.close()
    cols = ["rule_id", "title", "requirement_text", "usage_criteria_text",
            "tech_criteria_text", "notes", "item_no", "list_name"]
    matrix_rules = [dict(zip(cols, row)) for row in rows]
    print(f"  入力: {len(matrix_rules)} ルール")

    from pipeline.index.faiss_builder import build_matrix_rules_index
    result = build_matrix_rules_index(
        matrix_rules=matrix_rules,
        output_dir=FAISS_OUT,
        dry_run=DRY_RUN,
    )
    if result:
        ntotal, idx_path, meta_path = result
        print(f"✅ matrix_rules.index 構築完了: ntotal={ntotal}")

## 3. 品質チェック — 検索テスト

In [ ]:
# 構築したインデックスで簡易検索テストを実施
if not DRY_RUN:
    import faiss, numpy as np
    from sentence_transformers import SentenceTransformer

    MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    model = SentenceTransformer(MODEL)

    # entities 検索テスト
    idx_path = FAISS_OUT / "entities.index"
    meta_path = FAISS_OUT / "entities_meta.json"
    if idx_path.exists():
        index = faiss.read_index(str(idx_path))
        meta  = json.loads(meta_path.read_text())
        query = "Huawei Technologies"
        qv    = np.asarray(model.encode([query], normalize_embeddings=True), dtype="float32")
        D, I  = index.search(qv, 5)
        print(f"entities 検索テスト: '{query}'")
        for score, idx in zip(D[0], I[0]):
            if idx >= 0:
                print(f"  score={score:.3f}  {meta[idx]['entity_name']} ({meta[idx]['list_source']})")
else:
    print("[DRY_RUN] 検索テストをスキップ")

## サマリー

In [ ]:
print("=" * 50)
print("FAISS 構築サマリー")
print("=" * 50)
for f in sorted(FAISS_OUT.rglob("*")):
    if f.is_file():
        print(f"  {f.name}  ({f.stat().st_size:,} bytes)")
print()
print("次のノートブック → 05_export_to_fastapi.ipynb")